# Weight Decay (L2 Regularization)

L2 penalty **shrinks weights** toward zero, reducing overfitting.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)

class DummyDataGenerator:
    """Synthetic classification data for CPU-only tutorials."""
    def __init__(self, n_samples=256, n_features=8, n_classes=3, seed=42):
        g = torch.Generator().manual_seed(seed)
        self.X = torch.randn(n_samples, n_features, generator=g)
        self.y = torch.randint(0, n_classes, (n_samples,), generator=g)

    def tensors(self):
        return self.X, self.y

class SimpleMLP(nn.Module):
    def __init__(self, in_dim=8, hidden=16, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x):
        return self.net(x)


## 1. Train with and without weight decay

In [ ]:
def train_model(weight_decay=0.0, steps=150):
    gen = DummyDataGenerator(n_samples=200, n_features=8, n_classes=3)
    X, y = gen.tensors()
    model = SimpleMLP(hidden=64)
    opt = torch.optim.Adam(model.parameters(), lr=0.02, weight_decay=weight_decay)
    for _ in range(steps):
        opt.zero_grad()
        F.cross_entropy(model(X), y).backward()
        opt.step()
    return model

m_none = train_model(0.0)
m_l2 = train_model(0.1)

## 2. Weight histograms comparison

In [ ]:
w_none = m_none.net[0].weight.detach().flatten().numpy()
w_l2 = m_l2.net[0].weight.detach().flatten().numpy()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(w_none, bins=40, color='coral', edgecolor='black')
axes[0].set_title('No weight decay')
axes[1].hist(w_l2, bins=40, color='seagreen', edgecolor='black')
axes[1].set_title('weight_decay=0.1')
plt.tight_layout(); plt.show()

## 3. |weight| distribution overlay

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(np.abs(w_none), bins=40, alpha=0.6, label='no decay', color='coral')
ax.hist(np.abs(w_l2), bins=40, alpha=0.6, label='L2 decay', color='seagreen')
ax.legend(); ax.set_title('Absolute weight magnitudes')
plt.tight_layout(); plt.show()

## 4. L2 norm per layer

In [ ]:
def layer_norms(model):
    return [p.norm().item() for n, p in model.named_parameters() if 'weight' in n]

names = ['W1', 'W2']
n0 = layer_norms(m_none)
n1 = layer_norms(m_l2)
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(2); w = 0.35
ax.bar(x - w/2, n0, w, label='no decay', color='coral')
ax.bar(x + w/2, n1, w, label='weight decay', color='seagreen')
ax.set_xticks(x); ax.set_xticklabels(names); ax.legend()
ax.set_title('L2 norm of weight tensors')
plt.tight_layout(); plt.show()

## 5. Weight decay strengths sweep

In [ ]:
decs = [0.0, 0.01, 0.05, 0.1, 0.5]
norms = []
for d in decs:
    m = train_model(d, steps=80)
    norms.append(m.net[0].weight.norm().item())
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(decs, norms, 'o-', color='purple', lw=2)
ax.set_xlabel('weight_decay'); ax.set_ylabel('||W1||')
ax.set_title('Stronger L2 → smaller weights')
plt.tight_layout(); plt.show()